In [53]:
import uproot
import awkward as ak
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [54]:

ROOT_FILE = "/Users/lamanazer/Downloads/DTT_Kmunu_small.root"
TREE_NAME= "DecayTree"
N_EPOCHS= 40
EVENTS_PER_BATCH= 256
LEARNING_RATE= 1e-3
HIDDEN= 128
REJECTIONS =(0.90, 0.95, 0.99)
SELECT_REJ= 0.95    
TRAIN_SPLIT= 0.70
VALIDATION_SPLIT= 0.15
POOL_MODE = "sum"
K = 1
all_branches = ['Tr_AALLSAMEBPV', 'Tr_ACHI2DOCA', 'Tr_ADOCA', 'Tr_BPVIP','Tr_BPVIPCHI2', 'Tr_Charge', 'Tr_E', 'Tr_EcalE', 'Tr_Eta', 'Tr_HcalE','Tr_MinIP', 'Tr_MinIPChi2', 'Tr_ORIG_FLAGS', 
                'Tr_P', 'Tr_PIDK','Tr_PIDe', 'Tr_PIDmu', 'Tr_PIDp', 'Tr_PROBNNe', 'Tr_PROBNNk','Tr_PROBNNmu', 'Tr_PROBNNp', 'Tr_PROBNNpi', 'Tr_PT', 'Tr_PZ', 'Tr_Phi','Tr_PrsE',
                  'Tr_THETA', 'Tr_TRCHI2DOF', 'Tr_TRFITMATCHCHI2','Tr_TRFITTCHI2', 'Tr_TRFITVELOCHI2NDOF', 'Tr_TRGHOSTPROB', 'Tr_TRTYPE','Tr_TrFIRSTHITZ', 'Tr_TrFITTCHI2NDOF', 'Tr_VeloCharge', 'Tr_MC_ID']

tree = uproot.open(ROOT_FILE)[TREE_NAME]
arr = tree.arrays(all_branches, library="ak")

evt_sizes = ak.num(arr['Tr_AALLSAMEBPV'], axis=1).to_numpy()
if (evt_sizes == 0).any():
    keep = evt_sizes > 0
    arr = arr[keep]                       
    evt_sizes = evt_sizes[keep]           
data = {k: ak.to_numpy(ak.flatten(arr[k], axis=1)) for k in all_branches}

n_events = len(evt_sizes)
evt_borders = np.cumsum(np.concatenate(([0], evt_sizes)), dtype=np.int64)
tr_2_evt = np.repeat(np.arange(n_events), evt_sizes)
assert evt_borders[-1] == len(data['Tr_PT']), "something wrong here"
mc_id = data["Tr_MC_ID"]
orig_flags = data["Tr_ORIG_FLAGS"]




In [55]:
#signal definition 
y_kaon = ((np.abs(mc_id) == 321) & (orig_flags == 1))
y_kaon = y_kaon & (data["Tr_AALLSAMEBPV"] == 0)
y_kaon = y_kaon.astype(np.float32)

y_trad = (orig_flags == 1)
y_trad = y_trad & (data["Tr_AALLSAMEBPV"] == 0)
y_trad = y_trad.astype(np.float32)

In [ ]:
def build_event_labels(y_track_local):
    y_ev = np.zeros(n_events, dtype=np.float32)
    np.maximum.at(y_ev, tr_2_evt, y_track_local)
    return y_ev
y_event_kaon = build_event_labels(y_kaon)
y_event_trad = build_event_labels(y_trad)
LABELS = [("kaon", y_kaon, y_event_kaon),("trad", y_trad, y_event_trad),] #more efficient in later steps 


In [ ]:
leak = {'Tr_AALLSAMEBPV', 'Tr_ORIG_FLAGS', 'Tr_MC_ID'}
raw_cols = [c for c in all_branches if c not in leak]
feats = {c: np.nan_to_num(data[c].astype(np.float32),nan=0.0, posinf=0.0, neginf=0.0) for c in raw_cols} #if impact is low remove

pidk    = np.clip(feats['Tr_PIDK'],-10, 50)
probnnk = np.clip(feats['Tr_PROBNNk'],0,1)
ipchi2  = np.clip(feats['Tr_BPVIPCHI2'],0,100)
feats['pidk_x_probnnk'] = pidk * probnnk
feats['kaon_vs_muon']   = feats['Tr_PIDK'] - feats['Tr_PIDmu']
feats['kaon_vs_proton'] = feats['Tr_PIDK'] - feats['Tr_PIDp']
feats['probnnk_ratio']  = feats['Tr_PROBNNk'] / (feats['Tr_PROBNNpi'] + feats['Tr_PROBNNk'] + 1e-9)
feats['ip_x_probnnk']   = ipchi2 * probnnk


In [ ]:
feat_cols = list(feats.keys())
features = np.stack([feats[c] for c in feat_cols], axis=1)

print(f"tracks: {len(features):,}events: {n_events:,}  "f"features: {len(feat_cols)} (incl. {len(feat_cols)-len(raw_cols)} engineered)")
print(f"positive events (kaon):{int(y_event_kaon.sum()):,}({y_event_kaon.mean():.1%})")
print(f"positive events (trad): {int(y_event_trad.sum()):,} ({y_event_trad.mean():.1%})")

tracks: 11,566,653events: 257,620  features: 40 (incl. 5 engineered)
positive events (kaon):102,950 (40.0%)
positive events (trad): 188,450 (73.2%)


In [ ]:
def check(label_name, y_track_local, y_event_local):
    pos_events = np.where(y_event_local == 1)[0][:50]
    for e in pos_events:
        tracks_in_e = y_track_local[evt_borders[e]:evt_borders[e + 1]]
   

In [ ]:
check("kaon", y_kaon, y_event_kaon)
check("trad", y_trad, y_event_trad)

#Model
class Pool(nn.Module):
    def __init__(self, dim, mode="sum", K=1):
        super().__init__()
        assert mode in ("sum", "attention") #I ended up using sum since attention barely improved the final result
        self.mode, self.K = mode, K
        if mode== "attention":
            self.score = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(),nn.Linear(dim // 2, 1),)
    def _segsum(self, x, idx, n_ev):
        out = torch.zeros(n_ev, x.shape[1], device=x.device, dtype=x.dtype)
        out.index_add_(0, idx, x)
        return out

    def forward(self, x, idx, n_ev):
        if self.mode == "attention":
            s= self.score(x).squeeze(-1)
            m= torch.full((n_ev,), -1e30, device=x.device, dtype=x.dtype)
            m.scatter_reduce_(0, idx, s, reduce="amax")
            e= torch.exp(s - m[idx])
            z =torch.zeros(n_ev, device=x.device, dtype=x.dtype)
            z.index_add_(0, idx, e)
            a=(e / (z[idx] + 1e-12)).unsqueeze(-1)
        else:
            a = None
        def weighted(t):
            return t if a is None else t*a
        pooled = [self._segsum(weighted(x), idx, n_ev)]

        if self.K > 1:
            xb =torch.tanh(x)
            p =xb
            for _ in range(2, self.K + 1):
                p =p*xb
                pooled.append(self._segsum(weighted(p),idx,n_ev))
        return torch.cat(pooled, dim=1)

In [ ]:
class DeepSetsEvent(nn.Module):
    def __init__(self, in_dim, mode="sum", K=1, hidden=HIDDEN):
        super().__init__()
        self.phi = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),nn.Linear(hidden, hidden), nn.ReLU(),)
        self.pool = Pool(hidden, mode=mode, K=K)
        self.rho = nn.Sequential(nn.Linear(K * hidden, hidden), nn.ReLU(),nn.Linear(hidden, hidden // 2), nn.ReLU(),nn.Linear(hidden // 2, 1),)

    def forward(self, x, idx, n_events):
        x = self.phi(x)
        pooled = self.pool(x, idx, n_events)
        x = self.rho(pooled)
        return x.squeeze(-1)

In [ ]:
def eff_at_rejection(y_true, scores, rejection):
    bkg = scores[y_true==0]
    sig = scores[y_true==1]
    if len(bkg) == 0 or len(sig) ==0:
        return float("nan")
    thr = np.quantile(bkg, rejection)
    return float((sig > thr).mean())

In [ ]:
rng=np.random.default_rng(SEED)
perm=rng.permutation(n_events)

n_train= int(n_events*TRAIN_SPLIT)
n_val = int(n_events*(TRAIN_SPLIT + VALIDATION_SPLIT))
ev_train, ev_val,ev_test= perm[:n_train],perm[n_train:n_val], perm[n_val:]

def gather_tracks(ev_ids):
    return np.concatenate(
        [np.arange(evt_borders[e], evt_borders[e + 1]) for e in ev_ids])
tr_tracks=gather_tracks(ev_train)
va_tracks=gather_tracks(ev_val)
te_tracks=gather_tracks(ev_test)

In [ ]:
scaler = RobustScaler()
X =features.copy()
X[tr_tracks]=scaler.fit_transform(X[tr_tracks])
X[va_tracks]= scaler.transform(X[va_tracks])
X[te_tracks]= scaler.transform(X[te_tracks])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_t= torch.tensor(X,dtype=torch.float32,device=device)

In [ ]:
def make_batches(ev_ids):
    batches = []
    for s in range(0,len(ev_ids), EVENTS_PER_BATCH):
        chunk= ev_ids[s:s +EVENTS_PER_BATCH]
        sizes = evt_sizes[chunk]
        g_tracks = np.concatenate(np.arange(evt_borders[e], evt_borders[e + 1]) for e in chunk])
        local = torch.tensor(np.repeat(np.arange(len(chunk)), sizes),dtype=torch.long, device=device)
        g_tracks_t= torch.tensor(g_tracks, dtype=torch.long, device=device)
        ev_t =torch.tensor(chunk, dtype=torch.long, device=device)
        batches.append((g_tracks_t,local,ev_t,len(chunk)))
    return batches
train_batches = make_batches(ev_train)
val_batches = make_batches(ev_val)
test_batches = make_batches(ev_test)

In [ ]:
if train_batches and train_batches[-1][0].shape[0]<= 1:
    train_batches = train_batches[:-1]


In [ ]:
def train_one(label_name, y_event_local):
    torch.manual_seed(SEED)                 # same init per config
    run_rng = np.random.default_rng(SEED)   # same batch order too
    y_ev_t = torch.tensor(y_event_local, dtype=torch.float32, device=device)
    y_val_np = y_event_local[ev_val]
    y_test_np = y_event_local[ev_test]

    model = DeepSetsEvent(features.shape[1], mode=POOL_MODE, K=K).to(device)
    ytr= y_event_local[ev_train]
    pos_w = torch.tensor([(ytr == 0).sum()/max((ytr == 1).sum(), 1)],dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(),lr=LEARNING_RATE, weight_decay=1e-5)

    def predict(batches):
        model.eval()
        out = []
        with torch.no_grad():
            for g_tracks, local, ev_ids, n_ev in batches:logits = model(X_t[g_tracks], local, n_ev)
            out.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(out)

    best_val_eff, best_state, first_loss, first_auc= -1.0, None, None, None
    learned = True
    train_loss = None
    val_auc = None

    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss, nb = 0.0, 0
        for bi in run_rng.permutation(len(train_batches)):
            g_tracks, local, ev_ids, n_ev = train_batches[bi]
            opt.zero_grad()
            logits = model(X_t[g_tracks], local, n_ev)
            loss = loss_fn(logits, y_ev_t[ev_ids])
            loss.backward()
            opt.step()
            epoch_loss+=loss.item()
            nb+=1
        train_loss= epoch_loss/max(nb, 1)
        if first_loss is None:
            first_loss=train_loss
        val_scores = predict(val_batches)
        val_eff = eff_at_rejection(y_val_np,val_scores,SELECT_REJ)
        val_auc = roc_auc_score(y_val_np,val_scores)
        if first_auc is None:
            first_auc = val_auc

        if val_eff>best_val_eff:
            best_val_eff = val_eff
            best_state={k: v.detach().clone()
                          for k, v in model.state_dict().items()}
        if (epoch + 1)%5== 0:
            print(f"[{label_name}] epoch {epoch+1:2d}/{N_EPOCHS}"f"loss={train_loss:.4f}"f"val eff@{SELECT_REJ:.0%}={val_eff:.4f}"f"val AUC={val_auc:.4f}")

        if epoch== 9:
            loss_flat = abs(first_loss - train_loss)/ max(first_loss, 1e-9) < 0.01
            auc_near_chance =(val_auc - 0.5)< 0.02
            if loss_flat and auc_near_chance:
                print(f"  [{label_name}] loss flat and AUC problem "f"NOT LEARNED")
                learned = False
                break
    model.load_state_dict(best_state)
    test_scores = predict(test_batches)
    return {"label": label_name, "learned": learned,"test_auc": roc_auc_score(y_test_np, test_scores),"effs": {r: eff_at_rejection(y_test_np, test_scores, r)for r in REJECTIONS},"loss_start":first_loss,"loss_end":train_loss,"n_pos_test":int(y_test_np.sum()),"n_events_test": len(y_test_np),}

In [ ]:
POOL_MODE = "sum"
POOL_K= 1

In [72]:
results = []
for label_name, y_track_local, y_event_local in LABELS:
    print(f"\n'{label_name}'(pooling={POOL_MODE}, K={K})")
    results.append(train_one(label_name, y_event_local))


'kaon' (pooling=sum, K=1)
  [kaon] epoch  5/40  loss=0.8031  val eff@95%=0.1042  val AUC=0.6206
  [kaon] epoch 10/40  loss=0.8016  val eff@95%=0.1036  val AUC=0.6222
  [kaon] epoch 15/40  loss=0.8005  val eff@95%=0.1056  val AUC=0.6232
  [kaon] epoch 20/40  loss=0.7999  val eff@95%=0.1059  val AUC=0.6238
  [kaon] epoch 25/40  loss=0.7995  val eff@95%=0.1060  val AUC=0.6243
  [kaon] epoch 30/40  loss=0.7990  val eff@95%=0.1063  val AUC=0.6252
  [kaon] epoch 35/40  loss=0.7989  val eff@95%=0.1065  val AUC=0.6242
  [kaon] epoch 40/40  loss=0.7983  val eff@95%=0.1030  val AUC=0.6257

'trad' (pooling=sum, K=1)
  [trad] epoch  5/40  loss=0.3693  val eff@95%=0.0709  val AUC=0.5626
  [trad] epoch 10/40  loss=0.3689  val eff@95%=0.0702  val AUC=0.5644
  [trad] epoch 15/40  loss=0.3687  val eff@95%=0.0760  val AUC=0.5650
  [trad] epoch 20/40  loss=0.3686  val eff@95%=0.0754  val AUC=0.5670
  [trad] epoch 25/40  loss=0.3685  val eff@95%=0.0739  val AUC=0.5671
  [trad] epoch 30/40  loss=0.3683  v

In [73]:

header = (f"{'label':<8} {'learned':>8} {'ROC AUC':>9} "+ "".join(f"{f'eff{r:.0%}':>10}" for r in REJECTIONS)+ f" {'loss start->end':>18} {'pos rate':>14}")
print(header)
print("-"*len(header))
for r in results:
    flag = "yes" if r["learned"] else "NO"
    row = (f"{r['label']:<8} {flag:>8} {r['test_auc']:>9.4f} "+"".join(f"{r['effs'][rej]:>10.4f}" for rej in REJECTIONS)+ f" {r['loss_start']:>8.4f} -> {r['loss_end']:.4f}"+ f"   {r['n_pos_test']:,}/{r['n_events_test']:,}")
    print(row + ("did not learn"if not r["learned"] else ""))


label     learned   ROC AUC     eff90%    eff95%    eff99%    loss start->end       pos rate
--------------------------------------------------------------------------------------------
kaon          yes    0.6271     0.1874    0.0992    0.0241   0.8130 -> 0.7983   15,260/38,643
trad          yes    0.5612     0.1377    0.0667    0.0152   0.3716 -> 0.3682   28,177/38,643
